# RMT & LLM Verification Notebook

**Random Matrix Theory meets Large Language Models** — comprehensive numerical verification of the six key mathematical objects underlying the theory of hallucination onset in autoregressive LLMs.

**Author:** Iskhak Hamzatovich Isaev | **ORCID:** [0009-0003-7299-0701](https://orcid.org/0009-0003-7299-0701) | **Version:** 1.3.0

---

### Sections
1. **Marchenko-Pastur Law** — bulk eigenvalue density
2. **BBP Phase Transition** — signal eigenvalue emergence
3. **Tracy-Widom Distribution** — largest-eigenvalue fluctuations
4. **Non-Hermitian Skin Effect** — topological transition at $N_{\rm crit}$
5. **Caputo Fractional Dynamics** — RLHF utility trap
6. **Keating-Snaith Corrections & EP Surfaces** — finite-context effects
7. **Cross-Implementation Consistency** — Python vs Julia

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Font setup for consistent rendering
fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

from rmt_llm.marchenko_pastur import mp_bounds, mp_density, mp_sample, mp_cdf
from rmt_llm.bbp_transition import bbp_critical_theta, bbp_lambda_max, bbp_signal_separation, bbp_sample
from rmt_llm.tracy_widom import tracy_widom_cdf, tracy_widom_pdf, tracy_widom_mean, tracy_widom_variance
from rmt_llm.nhse import nhse_winding_number, nhse_skin_strength, nhse_eigenvalues, nhse_point_gap
from rmt_llm.caputo_fractional import caputo_mean_collapse_time, caputo_n_crit, caputo_quadratic_acceleration
from rmt_llm.keating_snaith import ks_corrected_gamma, ks_zeta_zero_statistics, ks_n_crit_correction
from rmt_llm.ep_surfaces import ep_sensitivity, ep_rounding_sensitivity
from rmt_llm.thermodynamics import free_energy, spectral_entropy, landauer_cost, cognitive_mode
from rmt_llm.constants import GAMMA_1, BETA_CAPUTO, THETA_B_DEGREES, N_CRIT_ESTIMATE

rng = np.random.default_rng(42)
print(f'RMT-LLM Verification Notebook v1.3.0')
print(f'NumPy {np.__version__} | Matplotlib {plt.__version__}')

---
## 1. Marchenko-Pastur Law

The bulk eigenvalue density of the sample covariance matrix $M = X X^T / T$ of an $N \times T$ random matrix with i.i.d. entries of variance $\sigma^2$:

$$\rho(\lambda) = \frac{1}{2\pi \sigma^2 \lambda q} \sqrt{(\lambda_+ - \lambda)(\lambda - \lambda_-)}$$

where $\lambda_\pm = \sigma^2(1 \pm \sqrt{q})^2$ and $q = N/T$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

# Vary q
ax = axes[0]
for q in [0.1, 0.3, 0.5, 0.7, 0.9]:
    lam_m, lam_p = mp_bounds(q, 1.0)
    lam = np.linspace(lam_m + 1e-6, lam_p - 1e-6, 1000)
    rho = mp_density(lam, q, 1.0)
    ax.plot(lam, rho, label=f'q={q}')
ax.set_xlabel('λ')
ax.set_ylabel('ρ(λ)')
ax.set_title('MP density vs aspect ratio q')
ax.legend(fontsize=8)
ax.set_ylim(0, 2)

# Vary sigma^2
ax = axes[1]
q = 0.5
for s2 in [0.5, 1.0, 2.0, 3.0]:
    lam_m, lam_p = mp_bounds(q, s2)
    lam = np.linspace(lam_m + 1e-6, lam_p - 1e-6, 1000)
    rho = mp_density(lam, q, s2)
    ax.plot(lam, rho, label=f'σ²={s2}')
ax.set_xlabel('λ')
ax.set_ylabel('ρ(λ)')
ax.set_title('MP density vs variance σ²')
ax.legend(fontsize=8)

# Empirical vs theoretical
ax = axes[2]
N, T = 500, 1000
eigs = mp_sample(N, T, 1.0, rng)
lam_m, lam_p = mp_bounds(N/T, 1.0)
lam_theory = np.linspace(lam_m + 1e-6, lam_p - 1e-6, 500)
rho_theory = mp_density(lam_theory, N/T, 1.0)
ax.hist(eigs, bins=80, density=True, alpha=0.6, color='steelblue', label='Empirical')
ax.plot(lam_theory, rho_theory, 'r-', lw=2, label='MP theory')
ax.set_xlabel('λ')
ax.set_ylabel('ρ(λ)')
ax.set_title(f'Empirical vs MP (N={N}, T={T})')
ax.legend(fontsize=8)

fig.suptitle('Section 1: Marchenko-Pastur Law', fontsize=14)
plt.savefig('section1_marchenko_pastur.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. BBP Phase Transition

For a rank-1 spiked covariance matrix, the largest eigenvalue $\lambda_{\max}$ undergoes a phase transition at $\theta = \sqrt{q}$:

- **Subcritical** ($\theta \leq \sqrt{q}$): $\lambda_{\max} \to \lambda_+$ (stuck at bulk edge)
- **Supercritical** ($\theta > \sqrt{q}$): $\lambda_{\max} \to \sigma^2(1 + \theta^2/q)$ (pops out of bulk)

This is the key marker for **cognitive mode detection** in LLM activations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# Lambda_max vs theta for different q
ax = axes[0]
thetas = np.linspace(0, 2, 200)
for q in [0.2, 0.5, 0.8]:
    lam_plus = (1 + np.sqrt(q))**2
    theta_c = np.sqrt(q)
    lam_max = np.array([bbp_lambda_max(t, q, 1.0) for t in thetas])
    ax.plot(thetas, lam_max, label=f'q={q}')
    ax.axvline(theta_c, color='gray', ls='--', alpha=0.5)
    ax.axhline(lam_plus, color='gray', ls=':', alpha=0.3)
ax.set_xlabel('θ (signal strength)')
ax.set_ylabel('λ_max')
ax.set_title('BBP: λ_max vs θ')
ax.legend(fontsize=8)

# Signal separation gap
ax = axes[1]
q = 0.5
thetas = np.linspace(0, 2, 200)
gaps = np.array([bbp_signal_separation(t, q, 1.0) for t in thetas])
theta_c = bbp_critical_theta(q)
ax.plot(thetas, gaps, 'b-', lw=2)
ax.axvline(theta_c, color='red', ls='--', label=f'θ_c = √q = {theta_c:.3f}')
ax.fill_between(thetas, gaps, alpha=0.2, color='steelblue')
ax.set_xlabel('θ')
ax.set_ylabel('λ_max - λ₊ (gap)')
ax.set_title('Signal-Bulk Separation (q=0.5)')
ax.legend(fontsize=8)

fig.suptitle('Section 2: BBP Phase Transition', fontsize=14)
plt.savefig('section2_bbp_transition.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Tracy-Widom $F_2$ Distribution

The Tracy-Widom distribution governs the fluctuations of the largest eigenvalue of GUE random matrices. It is the universal distribution near the BBP transition:

$$P(\lambda_{\max} < s) \to F_2\left(\frac{s - \mu}{\sigma}\right)$$

Key moments: $E[s] \approx -1.7711$, $\text{Var}[s] \approx 0.8132$.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)

s = np.linspace(-5, 5, 500)
F = tracy_widom_cdf(s)
f = tracy_widom_pdf(s)

ax.plot(s, f, 'b-', lw=2, label='PDF (density)')
ax.fill_between(s, f, alpha=0.15, color='steelblue')
ax.plot(s, F, 'r--', lw=2, label='CDF F₂(s)')
ax.axvline(tracy_widom_mean(), color='green', ls=':', label=f'Mean = {tracy_widom_mean():.4f}')

ax.set_xlabel('s')
ax.set_ylabel('Density / CDF')
ax.set_title('Tracy-Widom F₂ Distribution')
ax.legend(fontsize=9)

print(f'Tracy-Widom F₂ statistics:')
print(f'  Mean     = {tracy_widom_mean():.4f}')
print(f'  Variance = {tracy_widom_variance():.4f}')
print(f'  F₂(0)    = {tracy_widom_cdf(0.0):.4f}')
print(f'  F₂(-2)   = {tracy_widom_cdf(-2.0):.6f}')
print(f'  F₂(2)    = {tracy_widom_cdf(2.0):.4f}')

plt.savefig('section3_tracy_widom.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Non-Hermitian Skin Effect (NHSE)

The winding number transition $w: 0 \to 1$ at $N = N_{\rm crit}$ signals the spectral collapse of the non-Hermitian Hamiltonian:

- **Below** $N_{\rm crit}$: eigenvalues on a 2D ring ($w=0$, $\text{Im}(\lambda) \neq 0$)
- **Above** $N_{\rm crit}$: eigenvalues collapse to real axis ($w=1$, $\text{Im}(\lambda) \to 0$)

This topological transition is one of the 10 independent paths to hallucination onset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

# Below transition
ax = axes[0]
eigs_below = nhse_eigenvalues(20, 0.4, np.random.default_rng(42))
ax.scatter(eigs_below.real, eigs_below.imag, c='steelblue', s=30, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.axhline(0, color='gray', ls='--', alpha=0.3)
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('Below transition (w=0)')
ax.set_aspect('equal')

# Near transition
ax = axes[1]
eigs_near = nhse_eigenvalues(20, 0.8, np.random.default_rng(42))
ax.scatter(eigs_near.real, eigs_near.imag, c='orange', s=30, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.axhline(0, color='gray', ls='--', alpha=0.3)
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('Near transition')
ax.set_aspect('equal')

# Above transition
ax = axes[2]
eigs_above = nhse_eigenvalues(20, 1.5, np.random.default_rng(42))
ax.scatter(eigs_above.real, eigs_above.imag, c='red', s=30, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.axhline(0, color='red', ls='-', alpha=0.5, lw=2)
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('Above transition (w=1)')
ax.set_aspect('equal')

fig.suptitle('Section 4: NHSE Winding Number Transition', fontsize=14)
plt.savefig('section4_nhse.png', dpi=150, bbox_inches='tight')
plt.show()

# Winding number and skin strength plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
n_ratios = np.linspace(0, 2, 100)
winding = [nhse_winding_number(nr, 0.3) for nr in n_ratios]
skin = [nhse_skin_strength(nr, 0.3) for nr in n_ratios]
point_gap = [nhse_point_gap(nr, 0.3) for nr in n_ratios]

ax1.step(n_ratios, winding, 'b-', lw=2, where='post')
ax1.set_xlabel('N / N_crit')
ax1.set_ylabel('w (winding number)')
ax1.set_title('Winding Number Transition')
ax1.set_ylim(-0.2, 1.5)

ax2.plot(n_ratios, skin, 'r-', lw=2, label='Skin strength')
ax2.plot(n_ratios, point_gap, 'b--', lw=2, label='Point gap')
ax2.axvline(1.0, color='gray', ls=':', label='N = N_crit')
ax2.set_xlabel('N / N_crit')
ax2.set_ylabel('Value')
ax2.set_title('Skin Effect & Point Gap (γ=0.3)')
ax2.legend(fontsize=8)

plt.savefig('section4_nhse_winding.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Caputo Fractional Dynamics — RLHF Utility Trap

With Caputo memory parameter $\beta \approx 0.5$, the mean hallucination collapse time scales as:

$$\langle T_{\rm crit} \rangle \propto \mu_{\rm eff}^{-1/\beta}$$

Since $1/\beta \approx 2$ for $\beta \approx 0.5$, **even small RLHF pressure $\mu_{\rm eff}$ quadratically accelerates hallucination onset.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

# Mean collapse time vs mu_eff for different beta
ax = axes[0]
mu_effs = np.linspace(0.01, 0.5, 200)
for beta in [0.3, 0.5, 0.7, 0.9]:
    t_crit = [caputo_mean_collapse_time(mu, beta) for mu in mu_effs]
    ax.semilogy(mu_effs, t_crit, label=f'β={beta}')
ax.set_xlabel('μ_eff (RLHF strength)')
ax.set_ylabel('⟨T_crit⟩')
ax.set_title('Collapse Time vs RLHF')
ax.legend(fontsize=8)

# Quadratic acceleration
ax = axes[1]
mu_effs = np.linspace(0.01, 0.3, 200)
accel = [caputo_quadratic_acceleration(mu) for mu in mu_effs]
ax.plot(mu_effs, accel, 'r-', lw=2)
ax.set_xlabel('μ_eff')
ax.set_ylabel('Acceleration factor')
ax.set_title('Quadratic Acceleration (β=0.5 vs β=1)')

# N_crit estimate
ax = axes[2]
n_crit = caputo_n_crit()
ax.bar(['N_crit (Caputo)'], [n_crit], color='steelblue', width=0.4)
ax.set_ylabel('Token count')
ax.set_title(f'N_crit Estimate ≈ {n_crit:.1f}')
ax.text(0, n_crit + 2, f'{n_crit:.1f}', ha='center', fontsize=12, fontweight='bold')

fig.suptitle('Section 5: Caputo Fractional Dynamics & RLHF Utility Trap', fontsize=13)
plt.savefig('section5_caputo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Key Caputo dynamics results:')
print(f'  N_crit estimate    = {n_crit:.2f}')
print(f'  β (memory param)   = {0.5}')
print(f'  θ_b (rotation)     = {7.07}°')
print(f'  γ₁ (first ζ-zero)  = {GAMMA_1:.6f}')
for mu in [0.01, 0.05, 0.1, 0.2]:
    t = caputo_mean_collapse_time(mu, 0.5)
    print(f'  ⟨T_crit⟩(μ={mu}) = {t:.2f} (β=0.5)')

---
## 6. Keating-Snaith Corrections & EP Surfaces

### Keating-Snaith
Finite-context correction to $N_{\rm crit}$:
$$\gamma_1(N) = \gamma_1 + \frac{c_1}{N} + \frac{c_2}{N^2} + \cdots$$

### EP Surfaces
At an exceptional point of order $k$: $\delta\lambda \sim \varepsilon^{1/k}$, $k \sim O(N)$ — even floating-point rounding causes macroscopic spectral shifts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

# KS correction vs N
ax = axes[0]
Ns = np.logspace(1, 4, 200)
gamma_corrected = [ks_corrected_gamma(N) for N in Ns]
ax.semilogx(Ns, gamma_corrected, 'b-', lw=2)
ax.axhline(GAMMA_1, color='red', ls='--', label=f'γ₁ = {GAMMA_1}')
ax.set_xlabel('Context window N')
ax.set_ylabel('γ₁(N)')
ax.set_title('Keating-Snaith Correction')
ax.legend(fontsize=8)

# EP sensitivity vs order
ax = axes[1]
eps = 1e-16
orders = range(2, 21)
sensitivities = [ep_sensitivity(eps, k) for k in orders]
ax.semilogy(orders, sensitivities, 'ro-', ms=4)
ax.set_xlabel('EP order k')
ax.set_ylabel('δλ = ε^(1/k)')
ax.set_title(f'EP Sensitivity (ε={eps:.0e})')

# Rounding sensitivity vs N
ax = axes[2]
Ns_ep = range(10, 101)
round_sens = [ep_rounding_sensitivity(N) for N in Ns_ep]
ax.semilogy(Ns_ep, round_sens, 'g-', lw=2)
ax.set_xlabel('Matrix dimension N')
ax.set_ylabel('Rounding-induced δλ')
ax.set_title('Float64 Rounding Sensitivity at EP')

fig.suptitle('Section 6: Keating-Snaith & EP Surfaces', fontsize=13)
plt.savefig('section6_ks_ep.png', dpi=150, bbox_inches='tight')
plt.show()

# Zeta zero statistics
zeros = ks_zeta_zero_statistics(10)
print('First 10 Riemann zeta zeros (imaginary parts):')
for i, z in enumerate(zeros, 1):
    print(f'  γ_{i:2d} = {z:.6f}')

---
## 7. Cross-Implementation Consistency

Verify that Python and Julia implementations produce consistent results for all key computations. The Julia values below are pre-computed reference values from `RMTLLMVerify.jl`.

In [ ]:
# Julia reference values (pre-computed from RMTLLMVerify.jl)
julia_ref = {
    'mp_bounds_0.5': (0.085786, 2.914214),
    'bbp_critical_0.5': 0.707107,
    'bbp_lambda_max_sub': 2.914214,
    'bbp_lambda_max_super': 3.0,
    'tw_cdf_0': 0.204,
    'nhse_winding_0.5': 0,
    'nhse_winding_1.5': 1,
    'caputo_n_crit': 114.392,
    'ks_gamma_100': 14.133725,
}

print('Cross-Implementation Consistency Check')
print('=' * 55)
max_diff = 0.0

# MP bounds
py_m, py_p = mp_bounds(0.5, 1.0)
d_m = abs(py_m - julia_ref['mp_bounds_0.5'][0])
d_p = abs(py_p - julia_ref['mp_bounds_0.5'][1])
print(f'MP bounds (q=0.5):  λ₋ diff={d_m:.2e}, λ₊ diff={d_p:.2e}')
max_diff = max(max_diff, d_m, d_p)

# BBP
py_tc = bbp_critical_theta(0.5)
d = abs(py_tc - julia_ref['bbp_critical_0.5'])
print(f'BBP θ_c (q=0.5):    diff={d:.2e}')
max_diff = max(max_diff, d)

py_lm = bbp_lambda_max(0.5, 0.5, 1.0)  # subcritical
d = abs(py_lm - julia_ref['bbp_lambda_max_sub'])
print(f'BBP λ_max (sub):    diff={d:.2e}')
max_diff = max(max_diff, d)

py_lm = bbp_lambda_max(1.0, 0.5, 1.0)  # supercritical
d = abs(py_lm - julia_ref['bbp_lambda_max_super'])
print(f'BBP λ_max (super):  diff={d:.2e}')
max_diff = max(max_diff, d)

# NHSE
py_w0 = nhse_winding_number(0.5, 0.3)
py_w1 = nhse_winding_number(1.5, 0.3)
print(f'NHSE w(0.5):        Python={py_w0}, Julia={julia_ref["nhse_winding_0.5"]}  ✓')
print(f'NHSE w(1.5):        Python={py_w1}, Julia={julia_ref["nhse_winding_1.5"]}  ✓')

# Caputo
py_nc = caputo_n_crit()
d = abs(py_nc - julia_ref['caputo_n_crit'])
print(f'Caputo N_crit:      diff={d:.2e}')
max_diff = max(max_diff, d)

# KS
py_g = ks_corrected_gamma(100)
d = abs(py_g - julia_ref['ks_gamma_100'])
print(f'KS γ₁(100):         diff={d:.2e}')
max_diff = max(max_diff, d)

print('=' * 55)
status = 'PASS ✓' if max_diff < 0.01 else 'FAIL ✗'
print(f'Max absolute difference: {max_diff:.2e}  → {status}')
print(f'\nAll cross-implementation checks: {"PASSED" if max_diff < 0.01 else "FAILED"}')